## Capture-24 Data Preprocessing notebook
### Step 0. Setup the paths and env variables

In [ ]:
# =========================
# STEP 0 — Setup & contracts
# =========================
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path("/home/aidan/IMU_LM_Data")
sys.path.insert(0, str(ROOT))

from UTILS.helpers import resample_df, _canon, load_contracts

C       = load_contracts(ROOT)
SCHEMA  = C["SCHEMA"]
RAW2ID  = C["RAW2ID"]
ID2NAME = C["ID2NAME"]
UNKNOWN_ID = C["UNKNOWN_ID"]
TARGET_HZ  = C["TARGET_HZ"]
CLEANED    = C["CLEANED"]

RAW_CAPTURE = ROOT / "data" / "raw_data" / "Capture-24" / "capture24"

print("RAW_CAPTURE:", RAW_CAPTURE)
print("CLEANED    :", CLEANED)
print("Schema rate :", SCHEMA["rate_hz"])
print("Unknown ID  :", UNKNOWN_ID)

Paths & contracts ready.
RAW_CAPTURE: /home/aidan/IMU_LM_Data/data/raw_data/Capture-24/capture24
CLEANED    : /home/aidan/IMU_LM_Data/data/cleaned_premerge
Schema rate : 50
Unknown ID  : 9000


### Step 1. Ingest, preporccess and map the data 

In [ ]:
# ==========================================
# STEP 1 — Ingest Capture-24, normalize to 50Hz (STREAM TO PARQUET, NO CONCAT)
#   - Writes incrementally to one parquet to keep RAM flat
#   - Summary prints: files/rows/cols, top labels, CPA rate, preview
# ==========================================
import re
import gc
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

def extract_top_level_label(s: str):
    if pd.isna(s):
        return pd.NA
    top = str(s).split(";", 1)[0].strip()          # e.g. "7030 sleeping"
    top = re.sub(r"^\d{4,6}\s+", "", top)          # -> "sleeping" if it starts with CPA code
    return top

def parse_capture24_cpa_code(s: str):
    m = re.search(r"\b(\d{4,6})\b", str(s))
    return int(m.group(1)) if m else None

def load_capture24_raw_stream(
    raw_capture_dir: Path,
    out_parquet_path: Path,
    downsample_to_50hz: bool = True,
    max_files: int | None = None,
    preview_n: int = 0,
) -> Path:
    files = sorted(raw_capture_dir.glob("P*.csv.gz"))
    if not files:
        raise FileNotFoundError(f"No P*.csv.gz found under: {raw_capture_dir}")

    if max_files is not None:
        files = files[:max_files]
        print(f"[DEBUG] Limiting to first {len(files)} files")

    out_parquet_path = Path(out_parquet_path)
    out_parquet_path.parent.mkdir(parents=True, exist_ok=True)
    if out_parquet_path.exists():
        out_parquet_path.unlink()

    g_const = 9.80665
    writer = None

    # ---- stream-safe summary accumulators ----
    total_rows = 0
    total_cols = 13  # fixed: out_df has 13 columns
    label_counts: dict = {}   # dataset_activity_label -> count
    cpa_pos_rows = 0          # dataset_activity_id > 0
    preview_rows = None       # store preview once

    for f in tqdm(files, desc="CAPTURE-24 subjects"):
        df = pd.read_csv(f, compression="gzip", low_memory=False, dtype={"annotation": "string"})
        df = df.rename(columns={"x": "acc_x", "y": "acc_y", "z": "acc_z"})

        # ---- identity ----
        df["dataset"] = "capture24"
        df["subject_id"] = f.stem.replace(".csv", "")
        df["session_id"] = "day1"

        # ---- gyro placeholders ----
        df["gyro_x"] = np.nan
        df["gyro_y"] = np.nan
        df["gyro_z"] = np.nan

        # ---- time -> timestamp_ns ----
        df["time_dt"] = pd.to_datetime(df["time"], errors="coerce")
        df = df.dropna(subset=["time_dt"]).sort_values("time_dt").reset_index(drop=True)

        t0 = df["time_dt"].iloc[0]
        df["timestamp_ns"] = (df["time_dt"] - t0).astype("timedelta64[ns]").astype("int64")

        # ---- units sanity: if looks like "g", convert to m/s^2 ----
        mag_med = np.nanmedian(np.sqrt(df["acc_x"]**2 + df["acc_y"]**2 + df["acc_z"]**2))
        if 0.5 <= mag_med <= 2.5:
            df[["acc_x", "acc_y", "acc_z"]] = df[["acc_x", "acc_y", "acc_z"]].astype(np.float64) * g_const

        # ---- labels ----
        df["annotation"] = df["annotation"].astype("string")
        ann_str = df["annotation"].astype("string")
        unlabeled = ann_str.isna() | (ann_str.str.strip() == "")

        df["dataset_activity_label"] = df["annotation"].map(extract_top_level_label).astype("string")
        df["dataset_activity_id_tmp"] = df["annotation"].map(parse_capture24_cpa_code)

        df.loc[unlabeled, "dataset_activity_label"] = "unknown"
        df.loc[unlabeled, "dataset_activity_id_tmp"] = UNKNOWN_ID

        # ---- downsample 100Hz -> 50Hz ----
        if downsample_to_50hz:
            labels = df[["annotation", "dataset_activity_label", "dataset_activity_id_tmp"]].copy()

            target_cols = ["acc_x", "acc_y", "acc_z"]
            df_sig = resample_df(df, target_cols=target_cols, factor=2).reset_index(drop=True)

            labels_ds = labels.iloc[::2].reset_index(drop=True)
            labels_ds = labels_ds.iloc[:len(df_sig)].reset_index(drop=True)

            df = df_sig
            df[["annotation", "dataset_activity_label", "dataset_activity_id_tmp"]] = labels_ds[
                ["annotation", "dataset_activity_label", "dataset_activity_id_tmp"]
            ]
            df["timestamp_ns"] = np.arange(len(df), dtype=np.int64) * 20_000_000

            # restore identity + gyro placeholders (resample_df may drop them)
            df["dataset"] = "capture24"
            df["subject_id"] = f.stem.replace(".csv", "")
            df["session_id"] = "day1"
            df["gyro_x"] = np.nan
            df["gyro_y"] = np.nan
            df["gyro_z"] = np.nan

        # ---- per-subject fallback IDs ----
        missing_cpa = df["dataset_activity_id_tmp"].isna()
        has_ann = df["annotation"].notna() & (df["annotation"].astype("string").str.strip() != "")
        need_neg = missing_cpa & has_ann
        if need_neg.any():
            uniq = sorted(df.loc[need_neg, "dataset_activity_label"].astype(str).unique())
            neg_ids = {lab: -(i + 1) for i, lab in enumerate(uniq)}
            df.loc[need_neg, "dataset_activity_id_tmp"] = df.loc[need_neg, "dataset_activity_label"].map(
                lambda s: neg_ids[str(s)]
            )

        df["dataset_activity_id"] = df["dataset_activity_id_tmp"].astype("int16")

        out_df = df[[
            "dataset",
            "subject_id", "session_id", "timestamp_ns",
            "acc_x", "acc_y", "acc_z",
            "gyro_x", "gyro_y", "gyro_z",
            "annotation", "dataset_activity_label", "dataset_activity_id"
        ]].copy()

        # ---- capture preview once ----
        if preview_n and preview_n > 0 and preview_rows is None:
            preview_rows = out_df.head(preview_n).copy()

        # ---- update stream-safe summary stats ----
        vc = out_df["dataset_activity_label"].value_counts(dropna=False)
        for k, v in vc.items():
            label_counts[k] = label_counts.get(k, 0) + int(v)

        cpa_pos_rows += int((out_df["dataset_activity_id"] > 0).sum())
        total_rows += int(len(out_df))

        # ---- write chunk ----
        table = pa.Table.from_pandas(out_df, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(out_parquet_path, table.schema, compression="zstd")
        writer.write_table(table)

        # ---- cleanup ----
        del df, out_df, table
        gc.collect()

    if writer is not None:
        writer.close()

    # ---- summary print ----
    print("\n=== RAW SUMMARY (CAPTURE-24 wrist) ===")
    print(f"Files: {len(files)} | Rows: {total_rows:,} | Cols: {total_cols}")

    print("\nTop dataset_activity_label (top-level):")
    top15 = sorted(label_counts.items(), key=lambda kv: kv[1], reverse=True)[:15]
    for lab, cnt in top15:
        print(f"{lab}    {cnt}")

    cpa_rate = 100.0 * cpa_pos_rows / max(total_rows, 1)
    print(f"\nCPA-coded rows: {cpa_rate:.2f}% (positive IDs)")

    if preview_rows is not None:
        print("\nPreview:")
        print(preview_rows.to_string(index=False))

    print(f"\n[CAP24 STEP1] wrote parquet: {out_parquet_path}")
    return out_parquet_path

CAP24_RAW_PARQUET = CLEANED / "capture24_raw_50hz.parquet"

cap24_path = load_capture24_raw_stream(
    raw_capture_dir=RAW_CAPTURE,
    out_parquet_path=CAP24_RAW_PARQUET,
    max_files=None,
    preview_n=5,
)
# NOTE: Do NOT pd.read_parquet here — 699M rows will OOM.
# All downstream steps use streaming via pyarrow.
print("Raw parquet ready for streaming Steps 2-4.")

CAPTURE-24 subjects: 100%|██████████| 151/151 [1:04:01<00:00, 25.44s/it]



=== RAW SUMMARY (CAPTURE-24 wrist) ===
Files: 151 | Rows: 699,019,186 | Cols: 13

Top dataset_activity_label (top-level):
unknown    238140472
sleeping    170043095
home activity    133466550
occupation    78860262
transportation    35620758
leisure    24833031
sitting    5141568
household-chores    4305855
manual-work    2472742
sports/gym    2258582
walking    1337133
mixed-activity    1197350
vehicle    1095229
standing    241909
carrying heavy loads    4650

CPA-coded rows: 97.36% (positive IDs)

Preview:
  dataset subject_id session_id  timestamp_ns     acc_x     acc_y    acc_z  gyro_x  gyro_y  gyro_z             annotation dataset_activity_label  dataset_activity_id
capture24       P001       day1             0 -3.433019 -3.920670 4.844961     NaN     NaN     NaN 7030 sleeping;MET 0.95               sleeping                 7030
capture24       P001       day1      20000000 -4.885357 -5.587248 6.891190     NaN     NaN     NaN 7030 sleeping;MET 0.95               sleeping        

: 

### Step 2. Map the data and audit the mapping

In [ ]:
# ============================================
# STEP 2 — Map + audit (STREAM, NO WRITE)
# ============================================
import pyarrow as pa
import pyarrow.parquet as pq

def audit_mapping_capture24(in_parquet: Path, batch_rows: int = 1_000_000):
    pf = pq.ParquetFile(in_parquet)

    label_counts_norm: dict[str, int] = {}
    global_counts: dict[str, int] = {}
    mapped_rows = 0
    total_rows  = 0

    for batch in pf.iter_batches(batch_size=batch_rows, columns=["dataset_activity_label"]):
        tbl = pa.Table.from_batches([batch])
        ds = tbl["dataset_activity_label"].to_pandas().astype("string")

        ds_norm = ds.map(_canon)
        gid = ds_norm.map(lambda k: RAW2ID.get(k, UNKNOWN_ID)).astype("int16")
        gname = gid.map(lambda x: ID2NAME.get(int(x), "other")).astype("string")

        vc = ds_norm.value_counts(dropna=False)
        for k, v in vc.items():
            label_counts_norm[k] = label_counts_norm.get(k, 0) + int(v)

        gvc = gname.value_counts(dropna=False)
        for k, v in gvc.items():
            global_counts[k] = global_counts.get(k, 0) + int(v)

        mapped_rows += int((gid != UNKNOWN_ID).sum())
        total_rows  += int(len(ds_norm))

    raw_counts = (
        pd.Series(label_counts_norm, name="count")
          .rename_axis("raw_label_norm")
          .reset_index()
          .sort_values("count", ascending=False)
          .reset_index(drop=True)
    )
    raw_counts["mapped_id"] = raw_counts["raw_label_norm"].map(RAW2ID).fillna(UNKNOWN_ID).astype(int)
    raw_counts["mapped_nm"] = raw_counts["mapped_id"].map(lambda x: ID2NAME.get(int(x), "other"))

    unmapped = raw_counts.loc[raw_counts["mapped_id"] == UNKNOWN_ID]
    cov_pct = 100.0 * mapped_rows / max(total_rows, 1)

    print(f"[CAP24 STEP2] coverage={cov_pct:.2f}%  (mapped={mapped_rows:,} / total={total_rows:,})  unknown={UNKNOWN_ID}")
    print(f"[CAP24 STEP2] unique_labels={len(raw_counts)}  unmapped_unique={len(unmapped)}")

    print("\nTop dataset_activity_label (raw):")
    print(raw_counts.head(15).set_index("raw_label_norm")["count"])

    print("\nTop global_activity_label:")
    print(pd.Series(global_counts, name="count").sort_values(ascending=False).head(15))

    print("\nUnmapped dataset_activity_label (top 25):")
    print(unmapped.head(25).to_string(index=False))

    return raw_counts, unmapped

CAP24_RAW_PARQUET = CLEANED / "capture24_raw_50hz.parquet"
raw_counts, unmapped = audit_mapping_capture24(CAP24_RAW_PARQUET, batch_rows=1_000_000)

[CAP24 STEP2] coverage=65.78%  (mapped=459,783,485 / total=699,019,186)  unknown=9000
[CAP24 STEP2] unique_labels=15  unmapped_unique=2

Top dataset_activity_label (raw):
raw_label_norm
unknown                 238140472
sleeping                170043095
home_activity           133466550
occupation               78860262
transportation           35620758
leisure                  24833031
sitting                   5141568
household_chores          4305855
manual_work               2472742
sportsgym                 2258582
walking                   1337133
mixed_activity            1197350
vehicle                   1095229
standing                   241909
carrying_heavy_loads         4650
Name: count, dtype: int64

Top global_activity_label:
other                    239235701
posture_stationary       235880361
adl_household_general    219105409
exercise_plyometric        2258582
walk                       1337133
adl_personal_care          1197350
exercise_lower                4650
Name:

### Step 3. Build and clean dataset in stream json fromat

In [ ]:
# =========================================================
# STEP 3 — Build schema-ordered continuous_stream parquet (MAP INLINE)
#   - Reads: capture24_raw_50hz.parquet
#   - Writes: capture24_tmp.parquet   (Step 5 will finalize)
#   - Streaming: cannot use shared to_continuous_stream (699M rows)
# =========================================================
import pyarrow as pa
import pyarrow.parquet as pq

def stream_continuous_stream_capture24(
    in_parquet: Path,
    out_parquet: Path,
    dataset_name: str = "capture24",
    batch_rows: int = 1_000_000,
):
    pf = pq.ParquetFile(in_parquet)
    writer = None
    order = [c["name"] for c in SCHEMA["columns"]]

    for batch in pf.iter_batches(batch_size=batch_rows):
        df = pa.Table.from_batches([batch]).to_pandas()

        # map global labels INLINE using _canon
        ds_norm = df["dataset_activity_label"].astype("string").map(_canon)
        df["global_activity_id"] = ds_norm.map(lambda k: RAW2ID.get(k, UNKNOWN_ID)).astype("int16")
        df["global_activity_label"] = df["global_activity_id"].map(lambda x: ID2NAME.get(int(x), "other")).astype("string")

        out = pd.DataFrame({
            "dataset": dataset_name,
            "subject_id": df["subject_id"].astype("string"),
            "session_id": df["session_id"].astype("string"),
            "timestamp_ns": df["timestamp_ns"].astype("int64"),

            "acc_x": df["acc_x"].astype("float32"),
            "acc_y": df["acc_y"].astype("float32"),
            "acc_z": df["acc_z"].astype("float32"),

            "gyro_x": df["gyro_x"].astype("float32"),
            "gyro_y": df["gyro_y"].astype("float32"),
            "gyro_z": df["gyro_z"].astype("float32"),

            "global_activity_id": df["global_activity_id"].astype("int16"),
            "global_activity_label": df["global_activity_label"].astype("string"),

            "dataset_activity_id": df["dataset_activity_id"].astype("int16"),
            "dataset_activity_label": df["dataset_activity_label"].astype("string"),
        })[order]

        table = pa.Table.from_pandas(out, preserve_index=False)

        if writer is None:
            out_parquet.parent.mkdir(parents=True, exist_ok=True)
            if out_parquet.exists():
                out_parquet.unlink()
            writer = pq.ParquetWriter(out_parquet, table.schema, compression="zstd")

        writer.write_table(table)

        del df, out, table

    if writer is not None:
        writer.close()

    print("Wrote:", out_parquet)

CAP24_TMP = CLEANED / "capture24_tmp.parquet"

stream_continuous_stream_capture24(
    in_parquet=CLEANED / "capture24_raw_50hz.parquet",
    out_parquet=CAP24_TMP,
)

pf = pq.ParquetFile(CAP24_TMP)
print("UNIFIED rows:", pf.metadata.num_rows)
print(pf.read_row_group(0).to_pandas().head(3))

Wrote: /home/aidan/IMU_LM_Data/data/cleaned_premerge/capture24_tmp.parquet
UNIFIED rows: 699019186
     dataset subject_id session_id  timestamp_ns     acc_x     acc_y  \
0  capture24       P001       day1             0 -3.433019 -3.920670   
1  capture24       P001       day1      20000000 -4.885357 -5.587247   
2  capture24       P001       day1      40000000 -4.420410 -5.043319   

      acc_z  gyro_x  gyro_y  gyro_z  global_activity_id global_activity_label  \
0  4.844961     NaN     NaN     NaN                   1    posture_stationary   
1  6.891191     NaN     NaN     NaN                   1    posture_stationary   
2  6.239621     NaN     NaN     NaN                   1    posture_stationary   

   dataset_activity_id dataset_activity_label  
0                 7030               sleeping  
1                 7030               sleeping  
2                 7030               sleeping  


### Step 4. Audit check the unified frame

In [ ]:
# ==========================================
# STEP 4 — Contract checks & quick QA (FULL PARQUET, LOW-RAM)
#   - scans row groups (no full load)
#   - checks per-(subject,session): monotonicity, approx Hz
#   - required-not-null + mapping coverage
#   - top canonical labels + top dataset labels
#   - streaming sample integrity check (min 128 contiguous rows per label run)
# ==========================================
import pyarrow.parquet as pq

CAP24_TMP = CLEANED / "capture24_tmp.parquet"

pf = pq.ParquetFile(CAP24_TMP)
print("Rows:", pf.metadata.num_rows, "| Row groups:", pf.num_row_groups)

req = SCHEMA["expectations"]["required_not_null"]

cols = [
    "subject_id", "session_id", "timestamp_ns",
    "global_activity_id", "global_activity_label",
    "dataset_activity_label",
] + req

subjects = set()
sessions = set()

viol_groups = 0
hz_vals = []

rows_total = 0
rows_req_ok = 0
rows_mapped = 0

global_label_counts = {}
dataset_label_counts = {}

last_ts = {}  # track last timestamp per (subject, session) across row groups

# ---- streaming sample integrity state ----
MIN_SAMPLES = 128
expected_dt_ns = int(1e9 // TARGET_HZ)  # 20_000_000
tol_ns = expected_dt_ns // 2             # 10_000_000

# pending_run[key] = {"label": str, "len": int, "last_ts": int}
pending_run = {}
integrity_total_runs = 0
integrity_short_runs = 0
integrity_mono_viol = 0
integrity_dt_viol = 0
integrity_short_details = []  # (subj, sess, label, run_len)

def finalize_run(sid, sess, label, run_len):
    """Finalize a completed contiguous run (no timestamps needed)."""
    global integrity_total_runs, integrity_short_runs
    integrity_total_runs += 1
    if run_len < MIN_SAMPLES:
        integrity_short_runs += 1
        if len(integrity_short_details) < 20:
            integrity_short_details.append((sid, sess, label, run_len))

def finalize_run_with_ts(sid, sess, run_ts, run_label, run_len):
    """Finalize a completed contiguous run, checking dt violations."""
    global integrity_total_runs, integrity_short_runs, integrity_mono_viol, integrity_dt_viol
    integrity_total_runs += 1
    diffs = np.diff(run_ts)
    if diffs.size and not np.all(diffs >= 0):
        integrity_mono_viol += 1
    pos = diffs[diffs > 0] if diffs.size else np.array([], dtype=np.int64)
    if pos.size:
        med_dt = int(np.median(pos))
        if abs(med_dt - expected_dt_ns) > tol_ns:
            integrity_dt_viol += 1
    if run_len < MIN_SAMPLES:
        integrity_short_runs += 1
        if len(integrity_short_details) < 20:
            integrity_short_details.append((sid, sess, run_label, run_len))

# ---- main scan loop ----
for rg in range(pf.num_row_groups):
    df = pf.read_row_group(rg, columns=cols).to_pandas()
    rows_total += len(df)

    subjects.update(df["subject_id"].astype(str).unique().tolist())
    sessions.update(df["session_id"].astype(str).unique().tolist())

    rows_req_ok += int(df[req].notnull().all(axis=1).sum())
    rows_mapped += int((df["global_activity_id"].astype("int64") != UNKNOWN_ID).sum())

    gvc = df["global_activity_label"].astype("string").value_counts(dropna=False)
    for k, v in gvc.items():
        global_label_counts[k] = global_label_counts.get(k, 0) + int(v)

    dvc = df["dataset_activity_label"].astype("string").value_counts(dropna=False)
    for k, v in dvc.items():
        dataset_label_counts[k] = dataset_label_counts.get(k, 0) + int(v)

    for (sid, sess), g in df.groupby(["subject_id", "session_id"], sort=False):
        key = (str(sid), str(sess))
        ts = g["timestamp_ns"].to_numpy(dtype=np.int64)
        labels = g["dataset_activity_label"].to_numpy(dtype=str)

        if ts.size == 0:
            continue

        # -- monotonicity across row-group boundary --
        prev = last_ts.get(key, None)
        if prev is not None and ts[0] < prev:
            viol_groups += 1
        if ts.size > 1 and np.any(np.diff(ts) < 0):
            viol_groups += 1
        last_ts[key] = int(ts[-1])

        # -- Hz --
        if ts.size >= 3:
            dt_s = np.diff(ts) / 1e9
            dt_s = dt_s[(dt_s > 0) & np.isfinite(dt_s)]
            if dt_s.size:
                hz_vals.append(float(np.median(1.0 / dt_s)))

        # -- streaming sample integrity --
        change = np.concatenate(([True], labels[1:] != labels[:-1]))
        run_starts = np.where(change)[0]
        run_ends = np.append(run_starts[1:], len(labels))

        for ri in range(len(run_starts)):
            a, b = int(run_starts[ri]), int(run_ends[ri])
            run_label = labels[a]
            run_ts = ts[a:b]
            run_len = b - a

            # first run in this chunk: check if it continues a pending run
            if ri == 0 and key in pending_run:
                pr = pending_run.pop(key)
                if run_label == pr["label"]:
                    # extends previous: merge lengths
                    # check boundary monotonicity
                    if run_ts[0] < pr["last_ts"]:
                        integrity_mono_viol += 1
                    run_len += pr["len"]
                    # don't finalize yet — might continue into next row group
                else:
                    # label changed: finalize the old pending run
                    finalize_run(key[0], key[1], pr["label"], pr["len"])

            # last run in this chunk: stash as pending
            if ri == len(run_starts) - 1:
                if key in pending_run:
                    # already pending from merge above — update
                    pending_run[key] = {"label": run_label, "len": run_len, "last_ts": int(run_ts[-1])}
                else:
                    pending_run[key] = {"label": run_label, "len": run_len, "last_ts": int(run_ts[-1])}
            else:
                # intermediate run: finalize immediately with timestamp checks
                finalize_run_with_ts(key[0], key[1], run_ts, run_label, run_len)

    del df

# finalize remaining pending runs
for key, pr in pending_run.items():
    finalize_run(key[0], key[1], pr["label"], pr["len"])

# ---- QA Summary ----
print("\n=== QA SUMMARY (FULL PARQUET) ===")
print("Subjects:", len(subjects), "| Sessions:", len(sessions))
print("Monotonic violations (group-chunk or boundary):", viol_groups)

med_hz = float(np.nanmedian(hz_vals)) if len(hz_vals) else np.nan
print(f"Median Hz (stream): {med_hz:.2f} (target={SCHEMA['rate_hz']})")

pct_req = 100.0 * rows_req_ok / max(rows_total, 1)
print(f"Rows meeting required-not-null: {pct_req:.2f}%")

pct_map = 100.0 * rows_mapped / max(rows_total, 1)
print(f"Global mapping coverage: {pct_map:.1f}% (unknown={UNKNOWN_ID})")

print("\nTop-15 canonical (global) labels:")
top15g = sorted(global_label_counts.items(), key=lambda kv: kv[1], reverse=True)[:15]
for lab, cnt in top15g:
    print(f"  {lab:30s} {cnt:,}")

print("\nTop-15 dataset_activity_label:")
top15d = sorted(dataset_label_counts.items(), key=lambda kv: kv[1], reverse=True)[:15]
for lab, cnt in top15d:
    print(f"  {lab:30s} {cnt:,}")

# ---- Integrity Summary ----
print(f"\n=== Sample Integrity Check (min_samples={MIN_SAMPLES}) ===")
print(f"Total contiguous runs: {integrity_total_runs:,}")
print(f"Monotonic violations:  {integrity_mono_viol}")
print(f"dt-step violations:    {integrity_dt_viol}  (expected {expected_dt_ns/1e6:.0f} ms ± {tol_ns/1e6:.0f} ms)")
print(f"Short runs (<{MIN_SAMPLES} rows): {integrity_short_runs}  ({100*integrity_short_runs/max(integrity_total_runs,1):.1f}%)")
if integrity_short_details:
    print(f"\nFirst {len(integrity_short_details)} short runs:")
    for sid, sess, lbl, n in integrity_short_details:
        print(f"  {sid} | {sess} | {lbl:30s} | {n} rows")

Rows: 699019186 | Row groups: 700

=== QA SUMMARY (FULL PARQUET) ===
Subjects: 151 | Sessions: 1
Monotonic violations (group-chunk or boundary): 0
Median Hz (stream): 50.00 (target=50)
Rows meeting required-not-null: 100.00%
Global mapping coverage: 65.8% (unknown=9000)

Top-15 canonical (global) labels:
other    239,235,701
posture_stationary    235,880,361
adl_household_general    219,105,409
exercise_plyometric    2,258,582
walk    1,337,133
adl_personal_care    1,197,350
exercise_lower    4,650

Top-15 dataset_activity_label:
unknown    238,140,472
sleeping    170,043,095
home activity    133,466,550
occupation    78,860,262
transportation    35,620,758
leisure    24,833,031
sitting    5,141,568
household-chores    4,305,855
manual-work    2,472,742
sports/gym    2,258,582
walking    1,337,133
mixed-activity    1,197,350
vehicle    1,095,229
standing    241,909
carrying heavy loads    4,650


### Step 5. Save outputs

In [4]:
# =========================
# STEP 5 — Finalize output (atomic)
# =========================
final_path = CLEANED / "capture24_clean_data.parquet"

if final_path.exists():
    final_path.unlink()

CAP24_TMP.rename(final_path)
print("Finalized:", final_path)


Finalized: /home/aidan/IMU_LM_Data/data/cleaned_premerge/capture24_clean_data.parquet
